# Task 3: MERGE into Silver (Stretch Goal)

Deduplicate Bronze and MERGE into a Silver table that mirrors PostgreSQL. This is the core of Project 3.

## Step 1: Create Silver table schema

In [ ]:
spark.sql("""
  CREATE TABLE IF NOT EXISTS lakehouse.cdc.silver_customers (
    id INT, name STRING, email STRING, country STRING, last_updated_ms BIGINT
  ) USING iceberg
""")

print("Silver table created")

## Step 2: Read Bronze table and deduplicate

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.sql("SELECT * FROM lakehouse.cdc.bronze_customers")

bronze_with_key = bronze_df.withColumn(
  "entity_id", F.coalesce(F.col("after_id"), F.col("before_id"))
)

w = Window.partitionBy("entity_id").orderBy(F.col("ts_ms").desc())

deduped = bronze_with_key \
  .filter(F.col("op").isNotNull()) \
  .withColumn("rn", F.row_number().over(w)) \
  .filter("rn = 1").drop("rn")

print(f"Deduplicated records: {deduped.count()}")
deduped.show(5, truncate=False)

## Step 3: Create temp view for MERGE source

In [ ]:
deduped.createOrReplaceTempView("cdc_batch")
print("Temp view created")

## Step 4: Execute MERGE statement

In [ ]:
try:
    spark.sql("""
      MERGE INTO lakehouse.cdc.silver_customers AS t
      USING cdc_batch AS s
      ON t.id = s.entity_id
    
      WHEN MATCHED AND s.op = 'd' THEN DELETE
    
      WHEN MATCHED AND s.op IN ('c','u','r') THEN UPDATE SET
        t.name = s.after_name, t.email = s.after_email,
        t.country = s.after_country, t.last_updated_ms = s.ts_ms
    
      WHEN NOT MATCHED AND s.op != 'd' THEN INSERT
        (id, name, email, country, last_updated_ms)
        VALUES (s.entity_id, s.after_name, s.after_email, s.after_country, s.ts_ms)
    """)
    print("MERGE completed successfully")
except Exception as e:
    print(f"MERGE failed with error: {e}")
    print("\nUsing alternative approach with createOrReplace...")

## Step 4 (Alternative): Use createOrReplace if MERGE fails

In [ ]:
silver_data = deduped.select(
    F.col("entity_id").alias("id"),
    F.col("after_name").alias("name"),
    F.col("after_email").alias("email"),
    F.col("after_country").alias("country"),
    F.col("ts_ms").alias("last_updated_ms")
).filter(F.col("op") != "d")

silver_data.writeTo("lakehouse.cdc.silver_customers").createOrReplace()
print("Silver table populated via createOrReplace")

## Step 5: Validate Silver table

In [ ]:
print("Silver table count:")
spark.sql("SELECT count(*) FROM lakehouse.cdc.silver_customers").show()

print("\nSilver table sample (LIMIT 5):")
spark.sql("SELECT * FROM lakehouse.cdc.silver_customers ORDER BY id LIMIT 5").show(truncate=False)

## Step 6: Compare with PostgreSQL source

In [ ]:
print("Silver table ready for comparison with PostgreSQL source.")
print("\nExpected behavior:")
print("- If counts match, all CDC events captured and deduplicated")
print("- If Silver < PostgreSQL, simulator may have continued - re-run Bronze + MERGE to catch up")
print("- If you run MERGE again without new data, Silver rows remain unchanged (idempotent)")